In [ ]:
import yfinance as yf
import pandas as pd
import time
import os
import pickle
import random

# ================= 参数设置 =================
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']
start_year = 2013
end_year = 2025
chunk_size = 3  # 每次下载多少年，分段下载更稳
cache_dir = "yfinance_cache"
os.makedirs(cache_dir, exist_ok=True)
output_dir = "yfinance_output"
os.makedirs(output_dir, exist_ok=True)

# ================= 全局限速器 =================
LAST_TIME = 0
def wait_global(min_interval=6):
    global LAST_TIME
    now = time.time()
    if now - LAST_TIME < min_interval:
        time.sleep(min_interval - (now - LAST_TIME))
    LAST_TIME = time.time()

# ================= 下载函数 =================
def safe_download_chunk(ticker, start, end, max_retries=6):
    cache_file = os.path.join(cache_dir, f"{ticker}_{start}_{end}.pkl")
    # 缓存机制
    if os.path.exists(cache_file):
        print(f"✅ {ticker} {start}-{end} 使用缓存")
        with open(cache_file, 'rb') as f:
            return pickle.load(f)
    
    for attempt in range(max_retries):
        try:
            wait_global(6)
            print(f"⬇️ 下载 {ticker} {start}-{end} (尝试 {attempt+1})")
            df = yf.download(ticker, start=f"{start}-01-01", end=f"{end}-12-31", progress=False, threads=False)
            if df is None or df.empty:
                raise ValueError("数据为空")
            
            with open(cache_file, 'wb') as f:
                pickle.dump(df, f)
            
            print(f"✅ {ticker} {start}-{end} 下载成功")
            return df
        except Exception as e:
            wait_time = (2 ** attempt) + random.uniform(3,6)
            print(f"❌ {ticker} {start}-{end} 失败: {e}, 等待 {wait_time:.1f}s 重试")
            time.sleep(wait_time)
            if attempt >= 2:
                cooldown = random.uniform(30,60)
                print(f"🧊 连续失败，冷却 {cooldown:.1f}s")
                time.sleep(cooldown)
    print(f"🚫 {ticker} {start}-{end} 最终失败")
    return None

# ================= 技术指标 =================
def add_technical_indicators(df, ticker):
    df[f"{ticker}_SMA_20"] = df["Close"].rolling(20).mean()
    df[f"{ticker}_EMA_20"] = df["Close"].ewm(span=20, adjust=False).mean()
    
    delta = df["Close"].diff()
    up, down = delta.clip(lower=0), -delta.clip(upper=0)
    roll_up = up.rolling(14).mean()
    roll_down = down.rolling(14).mean()
    rs = roll_up / roll_down
    df[f"{ticker}_RSI_14"] = 100 - (100 / (1 + rs))
    
    exp1 = df["Close"].ewm(span=12, adjust=False).mean()
    exp2 = df["Close"].ewm(span=26, adjust=False).mean()
    df[f"{ticker}_MACD"] = exp1 - exp2
    df[f"{ticker}_MACD_Signal"] = df[f"{ticker}_MACD"].ewm(span=9, adjust=False).mean()
    
    df = df.rename(columns={
        'Open': f"{ticker}_Open",
        'High': f"{ticker}_High",
        'Low': f"{ticker}_Low",
        'Close': f"{ticker}_Close",
        'Volume': f"{ticker}_Volume",
        'Adj Close': f"{ticker}_AdjClose"
    })
    return df

# ================= 主循环 =================
all_stocks_data = []

for ticker in tickers:
    yearly_dfs = []
    year = start_year
    while year <= end_year:
        chunk_end = min(year + chunk_size - 1, end_year)
        df = safe_download_chunk(ticker, year, chunk_end)
        if df is not None:
            df = add_technical_indicators(df, ticker)
            csv_file = os.path.join(output_dir, f"{ticker}_{year}_{chunk_end}.csv")
            df.to_csv(csv_file)
            print(f"💾 保存 {csv_file}")
            yearly_dfs.append(df)
        # 段间随机休息 3~8 秒
        time.sleep(random.uniform(3,8))
        year = chunk_end + 1
    
    if yearly_dfs:
        stock_df = pd.concat(yearly_dfs)
        all_stocks_data.append(stock_df)
    # 每支股票下载完休息 30~60 秒
    time.sleep(random.uniform(30,60))

# ================= 合并所有股票 =================
merged_data = pd.concat(all_stocks_data, axis=1)
merged_data = merged_data.loc[:,~merged_data.columns.duplicated()]
merged_data.dropna(inplace=True)
merged_csv = os.path.join(output_dir, "all_stocks_2013_2025.csv")
merged_data.to_csv(merged_csv)
print(f"🎉 所有股票数据已合并保存: {merged_csv}")

# ================= 收盘价相关性 =================
close_cols = [f"{t}_Close" for t in tickers]
correlation_matrix = merged_data[close_cols].corr()
corr_csv = os.path.join(output_dir, "correlation_matrix.csv")
correlation_matrix.to_csv(corr_csv)
print("📊 收盘价相关性矩阵已保存:", corr_csv)
print(correlation_matrix)

⬇️ 下载 AAPL 2013-2015 (尝试 1)



1 Failed download:
['AAPL']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


❌ AAPL 2013-2015 失败: 数据为空, 等待 6.2s 重试
⬇️ 下载 AAPL 2013-2015 (尝试 2)



1 Failed download:
['AAPL']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


❌ AAPL 2013-2015 失败: 数据为空, 等待 6.5s 重试
⬇️ 下载 AAPL 2013-2015 (尝试 3)



1 Failed download:
['AAPL']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


❌ AAPL 2013-2015 失败: 数据为空, 等待 9.0s 重试
